In [1]:
import sys
import os

# Add the root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(project_root)

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy import stats
from src.utils import calculate_rsi, calculate_macd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [5]:
# Load dataset
df = pd.read_csv(r'..\..\data\bitcoin\btcusd_1-min_data.csv')
# Convert 'Timestamp' to datetime and set as index
df['Timestamp'] = pd.to_datetime(df['Timestamp'], unit='s')
df.set_index('Timestamp', inplace=True)
df = df.sort_index()
df.tail() 

,Open,High,Low,Close,Volume
Timestamp,,,,,
2025-08-02 21:50:00,112955.0,112955.0,112943.0,112944.0,0.369770
2025-08-02 21:51:00,112944.0,112944.0,112943.0,112944.0,0.009608
2025-08-02 21:52:00,112944.0,112956.0,112905.0,112917.0,1.211315
2025-08-02 21:53:00,112917.0,112971.0,112917.0,112971.0,0.366485
2025-08-02 21:54:00,112971.0,113025.0,112970.0,113025.0,2.309270


In [6]:
# Compute the Average price
df['Average_Price'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4

In [7]:
# Compute VWAP and Resample
VWP = df['Volume'] * df['Average_Price']
# Resample VWAP to daily, monthly, yearly, and quarterly frequency
vwap_minutely = VWP.resample('min').sum() / df['Volume'].resample('min').sum()
vwap_daily = VWP.resample('D').sum()/ df['Volume'].resample('D').sum()
vwap_monthly = VWP.resample('ME').sum()/ df['Volume'].resample('ME').sum()
vwap_yearly = VWP.resample('YE-DEC').sum()/ df['Volume'].resample('YE-DEC').sum()
vwap_quarterly = VWP.resample('QE-DEC').sum()/ df['Volume'].resample('QE-DEC').sum()

df_minute = vwap_minutely.to_frame(name='Weighted_Price')
df_daily = vwap_daily.to_frame(name='Weighted_Price')
df_monthly = vwap_monthly.to_frame(name='Weighted_Price')
df_yearly = vwap_yearly.to_frame(name='Weighted_Price')
df_quarterly = vwap_quarterly.to_frame(name='Weighted_Price')

AR model: The current value depend of the series depend on the past value itself.
MA model: The current value depend on the past forecast errors
ARMA model: Combination of AR and MA

In [8]:
# Fit AR model to daily data
from statsmodels.tsa.ar_model import AutoReg
model = AutoReg(df_daily['Weighted_Price'].dropna(), lags=3)
model_fit = model.fit()
print('Coefficients: %s' % model_fit.params)

Coefficients: const                9.289624
Weighted_Price.L1    1.204431
Weighted_Price.L2   -0.232641
Weighted_Price.L3    0.028707
dtype: float64


C:\Users\miraz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


Here, const is the intercept: Baseline adjustment
L1 this is coefficient for the first lag: 1.204431 (>1) the series has strong persistance
L2 this is coefficent for the second lag: -0.232641 (Increase in the current prediction by 1 unit the current prediction decrease by about 0.232641 unit)
....
The current  bitcoin price depends heavily on the last price and lasser extent on L2 and L3.

In [10]:
# Fit AR model to daily data
from statsmodels.tsa.ar_model import AutoReg
model = AutoReg(df_daily['Weighted_Price'].dropna(), lags=12)
model_fit = model.fit()
print('Coefficients: %s' % model_fit.params)

Coefficients: const                 9.635332
Weighted_Price.L1     1.206798
Weighted_Price.L2    -0.226326
Weighted_Price.L3    -0.030935
Weighted_Price.L4     0.110950
Weighted_Price.L5    -0.100120
Weighted_Price.L6     0.058333
Weighted_Price.L7    -0.030276
Weighted_Price.L8    -0.030990
Weighted_Price.L9     0.086816
Weighted_Price.L10   -0.011949
Weighted_Price.L11   -0.022179
Weighted_Price.L12   -0.009679
dtype: float64


C:\Users\miraz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [12]:
# Fit ARIMA model with a simpler order and drop missing values
from statsmodels.tsa.arima.model import ARIMA

data = df_daily['Weighted_Price'].dropna()

# Try a simpler ARIMA order
model = ARIMA(data, order=(2, 0, 0))
model_fit = model.fit()
print(model_fit.summary())

C:\Users\miraz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\miraz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
C:\Users\miraz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g

                               SARIMAX Results                                
Dep. Variable:         Weighted_Price   No. Observations:                 4958
Model:                 ARIMA(2, 0, 0)   Log Likelihood              -39885.824
Date:                Thu, 14 Aug 2025   AIC                          79779.648
Time:                        01:36:29   BIC                          79805.683
Sample:                             0   HQIC                         79788.777
                               - 4958                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.913e+04      5.648   3387.182      0.000    1.91e+04    1.91e+04
ar.L1          1.2000      0.006    197.525      0.000       1.188       1.212
ar.L2         -0.2002      0.006    -33.001      0.0